## **Этапы 1, 2**

In [ ]:
from datetime import datetime

# Инициализируем класс Account
class Account:

    # Добавляем флаг запрета на использование отрицательного баланса
    _allow_negative_balance = False

    # Описываем конструктор класса
    def __init__(self, account_holder:str, balance:float = 0) -> None:
        if balance < 0 and not self._allow_negative_balance:
            raise ValueError('Начальный баланс не может быть отрицательным')

        self.holder = account_holder
        self.operations_history = []
        self._balance = float(balance)

   # Описываем метод добавления операции в историю
    def operations_history_append(
        self,
        type_operation:str,
        amount:float,
        status:str
    ) -> None:
      self.operations_history.append({
        'type': type_operation,
        'amount': amount,
        'datetime': datetime.now(),
        'balance': self._balance,
        'status': status,
      })


    # Описываем метод пополнения банковского счета
    def deposit(self, amount:float) -> None:
        if amount <= 0:
            raise ValueError("Cумма пополнения должна быть положительной")

        self._balance += amount
        self.operations_history_append('deposit', amount, 'success')
        print(f"{amount} успешно внесены")


    # Описываем метод снятия денежных средств
    def withdraw(self, amount:float) -> None:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной!")

        if amount > self._balance:
            self.operations_history_append('withdraw', amount, 'fail')
            print("Недостаточно средств")

        else:
            self._balance -= amount
            self.operations_history_append('withdraw', amount, 'success')
            print(f"{amount} успешно сняты")


    # Получаем баланс
    def get_balance(self) -> float:
        return self._balance


    # Получаем копию операций (копия нужна для обеспечения безопасности)
    def get_history(self) -> list[dict]:
        return [op.copy() for op in self.operations_history]

# **Этап 3**

In [ ]:
# Инициализируем дочерний класс CreditAccount
class CreditAccount(Account):

    # Переопределяем флаг для разрешения использования отрицательного баланса
    _allow_negative_balance = True


    def __init__(
        self,
        account_holder:str,
        balance:float = 0,
        credit_limit:float = 0
    ) -> None:
        if credit_limit < 0:
            raise ValueError('Кредитный лимит не может быть отрицательным')

        if balance < -credit_limit:
            raise ValueError('Баланс не может быть меньше кредитного лимита')

        super().__init__(account_holder, balance)
        self.credit_limit = float(credit_limit)


    # Переопределяем метод добавления в историю
    def operations_history_append(
        self,
        type_operation:str,
        amount:float,
        status:str
    ) -> None:
        credit_used = (
            type_operation == 'withdraw'
            and status == 'success'
            and self._balance < 0
    )
        self.operations_history.append({
            'type': type_operation,
            'amount': amount,
            'datetime': datetime.now(),
            'balance': self._balance,
            'status': status,
            'credit_used': credit_used
        })


    # Описываем метод снятия денежных средств с кредитного счета
    def withdraw(self, amount:float) -> None:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной!")


        if amount > self._balance + self.credit_limit:
            self.operations_history_append('withdraw', amount, 'fail')
            print("Недостаточно средств (с учетом кредитного лимита)")
            return


        self._balance -= amount
        self.operations_history_append('withdraw', amount, 'success')
        print(f"{amount} успешно сняты!")


    # Получаем кредитный лимит
    def get_credit_limit(self) -> float:
        return self.credit_limit


    # Получаем доступные кредитные средства
    def available_credit(self) -> float:
        return self._balance + self.credit_limit